# ANALYSE-06 — Sensibilité 2026 : notre lac face aux familles d'openai/math (companion natif)

Sous-série **ANALYSE** (gradation #17545) — Pli 2 de la digestion **Origami** (#19898) : les résultats de combinatoire booléenne publiés par `openai/math` en 2026, lus depuis **notre** lac `Sensitivity` — le portage du théorème de Huang (2019).

Ce carnet est un companion **natif** : kernel Lean 4 sous WSL, chaque `#check` interroge le lac réellement compilé, chaque `#eval` calcule réellement. Le calcul numérique se fait **en Lean natif** — pas de repli Python, le suffixe du nom suit la convention de la série.

**Prérequis** : le tutoriel de la série principale (carnets 1 à 6). La lecture de [Lean-12b — Théorème de sensibilité de Huang](../Lean-12b-Lean-Sensitivity-Theorem.ipynb) rend la section 1 plus confortable, mais n'est pas exigée : nous rappelons ce dont nous avons besoin.

**Plan** : le socle nôtre (Huang) — les quatre familles 2026 citées avec leur statut — nos propres séparations sensibilité/sensibilité de bloc (module `Sensitivity.Separations`) vérifiées et calculées — trois exercices.

## Navigation

| Notebook précédent | Notebook suivant |
|---|---|
| [ANALYSE-04 — Trois primitives de PFR](ANALYSE-04-PFR-Primitives-Python.ipynb) | [ANALYSE-06 — version anglaise](ANALYSE-06-Sensibilite-2026-Lean_en.ipynb) |

## 0. Statut épistémique — la règle de ce carnet

La digestion Origami exige un **statut déclaré** pour chaque énoncé. Trois statuts existent : **nôtre** (prouvé dans notre lac), **formalisée** (prouvée par les auteurs d'openai/math, squelette Lean publié), **non formalisée** (aucune preuve formelle). La règle : jamais « OpenAI a prouvé X, donc X » — ce carnet distingue à chaque ligne ce que **nous prouvons et exécutons** de ce que **nous citons**.

| Objet | Statut | Où le vérifier |
|---|---|---|
| Théorème de Huang (2019) | **nôtre** | `#check` + `#print axioms`, section 1 |
| Familles 127, 132, 186, 192 (openai/math, 2026) | **formalisée** (par les auteurs) | citées section 2 — squelettes `lean/docs/NNN.md` du dépôt openai/math ; leurs preuves ne sont PAS exécutées ici |
| `sensitivity_le_blockSensitivity`, `orOfAnds_strict_separation`, `or3_blockSensitivity` | **nôtre** | `#check` + `#print axioms` + `#eval`, section 3 |

Ce que ce carnet **ne fait pas** : re-prouver les résultats 2026. Leur échelle (des centaines de pages par manuscrit) met la re-preuve hors de portée de ce dépôt — c'est précisément pourquoi le statut « formalisée » existe et pourquoi les squelettes restent la référence.

## 1. Le socle : Huang 2019, prouvé chez nous

Le point de départ du lac `Sensitivity` est le théorème de Huang (2019) et sa résolution de la conjecture de sensibilité : dans l'hypercube $Q_{m+1}$, si l'on colorie plus de la moitié des sommets ($|H| \ge 2^m + 1$), alors l'un des sommets coloriés a au moins $\sqrt{m+1}$ voisins coloriés. La conséquence centrale pour nous : la sensibilité de bloc d'une fonction booléenne est bornée par le **carré** de sa sensibilité, $bs(f) \le s(f)^2$.

Importons le lac. Ses globs couvrent les **sous-modules** (pas la racine) : nous importons donc explicitement les modules que ce carnet utilise — l'agrégateur racine `Sensitivity.lean` n'est pas un module compilé du lac :

In [1]:
import Sensitivity.Hypercube
import Sensitivity.MainTheorem
import Sensitivity.Fourier
import Sensitivity.Separations

open Sensitivity

import Sensitivity.Hypercube
import Sensitivity.MainTheorem
import Sensitivity.Fourier
import Sensitivity.Separations

open Sensitivity
--% env 0

Raw input:
{"cmd": "import Sensitivity.Hypercube\nimport Sensitivity.MainTheorem\nimport Sensitivity.Fourier\nimport Sensitivity.Separations\n\nopen Sensitivity"}
Raw output:
{"env": 0}

In [2]:
#check Sensitivity.huang_degree_theorem
#check Sensitivity.exists_eigenvalue

#check Sensitivity.huang_degree_theorem
──────▶  Sensitivity.huang_degree_theorem {m : ℕ} (H : Set (Q m.succ)) (hH : H.toFinset.card ≥ 2 ^ m + 1) :
  ∃ q ∈ H, √(↑m + 1) ≤ ↑(H ∩ q.adjacent).toFinset.card
#check Sensitivity.exists_eigenvalue
──────▶  Sensitivity.exists_eigenvalue {m : ℕ} (H : Set (Q m.succ)) (hH : H.toFinset.card ≥ 2 ^ m + 1) :
  ∃ y ∈ Submodule.span ℝ (e '' H) ⊓ (g m).range, y ≠ 0
--% env 1

Raw input:
{"cmd": "#check Sensitivity.huang_degree_theorem\n#check Sensitivity.exists_eigenvalue", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Sensitivity.huang_degree_theorem {m : ℕ} (H : Set (Q m.succ)) (hH : H.toFinset.card ≥ 2 ^ m + 1) :\n  ∃ q ∈ H, √(↑m + 1) ≤ ↑(H ∩ q.adjacent).toFinset.card"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sensitivity.exists_eigenvalue {m : ℕ} (H : Set (Q m.succ)) (hH : H.toFinset.card ≥ 2 ^ m + 1) :\n  ∃ y ∈ Submodule.span ℝ (e '' H) ⊓ (g m).range, y ≠ 0"}],
 "env": 1}

### Lecture du résultat

`huang_degree_theorem` est l'énoncé porteur : la borne spectrale $d^2$ sur le degré du sous-graphe induit. `exists_eigenvalue` en est le lemme spectral pivot (existence d'une valeur propre de module $\sqrt{d}$ pour l'opérateur $f_n$ de Huang sur l'espace libre $V_n$). L'arc complet — espace $V_n$, opérateur $f_n$, nilpotence, théorème principal — est parcouru dans [Lean-12b](../Lean-12b-Lean-Sensitivity-Theorem.ipynb) ; ici nous prenons le théorème comme socle et vérifions son empreinte logique :

In [3]:
#print axioms Sensitivity.huang_degree_theorem

#print axioms Sensitivity.huang_degree_theorem
──────▶  'Sensitivity.huang_degree_theorem' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 2

Raw input:
{"cmd": "#print axioms Sensitivity.huang_degree_theorem", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'Sensitivity.huang_degree_theorem' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 2}

### Sans `sorry`, empreinte minimale

`[propext, Classical.choice, Quot.sound]` — les trois axiomes standards de Mathlib, aucun `sorryAx`, aucun `native_decide`. Le théorème phare est **nôtre** au sens plein : prouvé dans le lac, vérifiable par cette commande à tout instant.

## 2. Ce qui a bougé en 2026 — quatre familles, citées avec leur statut

Le dépôt `openai/math` publie 242 familles de résultats avec squelettes Lean (`lean/docs/NNN.md`). Notre carte de résonance (distillation Origami, `30-resonance.md`) en identifie quatre dont les objets sont **littéralement les nôtres** — sensibilité, influences, coefficients de Fourier, degré. Statut des quatre : **formalisée** — par les auteurs, pas par nous. Nous citons leurs énoncés (repris de leurs résumés) ; leurs preuves ne sont pas exécutées ici.

| Famille | Énoncé (résumé fidèle) | Lien avec notre lac |
|---|---|---|
| **127** | La sensibilité **moyenne** d'une fonction à seuil polynomiale de degré $\le d$ sur le cube uniforme est $\le 8d\sqrt{n}$, pour $1 \le d \le n$ — la conjecture asymptotique de Gotsman–Linial | L'analogue « lisse » de notre sensibilité locale : même objet, moyenné sur le cube |
| **132** | Construit des fonctions booléennes totales avec $bs(f) \ge s(f)^\alpha$ pour un $\alpha > 2$ fixe, réfutant le renforcement quadratique de la conjecture de sensibilité | La question suivante naturelle sur **nos** objets $s$ et $bs$ — c'est la section 2.1 |
| **186** | Preuve des conjectures de largeur de seuil de Friedgut–Kalai : toute propriété croissante non triviale invariante par relabelling passe de $\varepsilon$ à $1-\varepsilon$ en largeur $O((\log n)^{-2})$ (graphes), $O_r((\log n)^{-r/(r-1)})$ (hypergraphes $r$-uniformes) | Le pont vers les influences — le vocabulaire $\hat f$, influence de la section 4 |
| **192** | Réfute la borne en racine liant coefficients de Fourier linéaires et degré : pour tout $C > 0$ il existe $f$ à valeurs signes avec $\sum_i \widehat f(\{i\}) > C\sqrt{\deg f}$ | Mêmes coefficients $\widehat f$ que notre module `Fourier` (section 4) |

Deux de ces familles sont des **réfutations** (132, 192) : la mode 2026 n'est pas seulement de prouver des bornes, mais de montrer que des bornes **naturelles** sont fausses. C'est exactement le geste que nous reproduisons à notre échelle en section 3 : exhiber un contre-exemple concret à l'égalité $s = bs$.

### 2.1 Famille 132 — la question suivante naturelle après Huang

Huang donne $bs(f) \le s(f)^2$ (pour les variantes globales de $s$ et $bs$). La question ouverte devient : **l'exposant 2 est-il le bon ?** Peut-on le renforcer ? La famille 132 répond par un contre-exemple : elle construit des fonctions où la sensibilité de bloc dépasse **toute borne renforcée** — au sens précis de ses définitions, que nous ne retranscrivons pas ici (l'énoncé exact vit dans son manuscrit et son squelette `lean/docs/132.md`).

Ce que **nous** prouvons, et qui est le socle pedagogique de cette question : la séparabilité est **réelle** — il existe des fonctions et des points où $s(f, x) < bs(f, x)$ strictement. C'est l'objet de la section 3 :

In [4]:
#check Sensitivity.sensitivity_le_blockSensitivity
#check Sensitivity.orOfAnds_strict_separation

#check Sensitivity.sensitivity_le_blockSensitivity
──────▶  Sensitivity.sensitivity_le_blockSensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : sensitivity F x ≤ blockSensitivity F x
#check Sensitivity.orOfAnds_strict_separation
──────▶  Sensitivity.orOfAnds_strict_separation :
  (sensitivity orOfAnds fun x => false) < blockSensitivity orOfAnds fun x => false
--% env 3

Raw input:
{"cmd": "#check Sensitivity.sensitivity_le_blockSensitivity\n#check Sensitivity.orOfAnds_strict_separation", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Sensitivity.sensitivity_le_blockSensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : sensitivity F x ≤ blockSensitivity F x"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sensitivity.orOfAnds_strict_separation :\n  (sensitivity orOfAnds fun x => false) < blockSensitivity orOfAnds fun x => false"}],
 "env": 3}

### Lecture — ce que nous avons prouvé

- `sensitivity_le_blockSensitivity` : retourner une coordonnée, c'est retourner un bloc singleton — donc $s(f, x) \le bs(f, x)$ **toujours**. C'est la borne plancher : les deux quantités ne peuvent pas se croiser.
- `orOfAnds_strict_separation` : sur un exemple explicite en dimension 4, l'inégalité est **stricte** — $s < bs$ en un point concret. L'écart entre $s$ et $bs$ n'est pas une curiosité théorique : c'est un phénomène calculable de bout en bout, et nous allons le calculer.

## 3. Nos définitions — le module `Sensitivity.Separations`

Le module ajouté par le Pli 2 définit les deux quantités et l'exemple séparateur. Toutes vivent dans le lac (namespace `Sensitivity`) :

In [5]:
#check Sensitivity.sensitivity
#check Sensitivity.blockSensitivity
#check Sensitivity.flipBlock
#check Sensitivity.orOfAnds

#check Sensitivity.sensitivity
──────▶  Sensitivity.sensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : ℕ
#check Sensitivity.blockSensitivity
──────▶  Sensitivity.blockSensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : ℕ
#check Sensitivity.flipBlock
──────▶  Sensitivity.flipBlock {n : ℕ} (x : Q n) (S : Finset (Fin n)) : Q n
#check Sensitivity.orOfAnds
──────▶  Sensitivity.orOfAnds (x : Q 4) : Bool
--% env 4

Raw input:
{"cmd": "#check Sensitivity.sensitivity\n#check Sensitivity.blockSensitivity\n#check Sensitivity.flipBlock\n#check Sensitivity.orOfAnds", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "Sensitivity.sensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : ℕ"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sensitivity.blockSensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : ℕ"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Sensitivity.flipBlock {n : ℕ} (x : Q n) (S : Finset (Fin n)) : Q n"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data": "Sensitivity.orOfAnds (x : Q 4) : Bool"}],
 "env": 4}

### Lecture des définitions

Sur le cube $Q_n = (\mathbb{Z}/2)^n$ (type `Q n = Fin n → Bool` dans le lac) :

- **`sensitivity F x`** — la sensibilité locale : le nombre de coordonnées $i$ dont le retournement **seul** change la valeur, $s(F, x) = \#\{i : F(x \oplus e_i) \ne F(x)\}$ ;
- **`flipBlock x S`** — retourner un **bloc** $S$ de coordonnées d'un coup : $(x \oplus \chi_S)$ ;
- **`blockSensitivity F x`** — la sensibilité de bloc locale, **définition standard de Nisan–Szegedy** : le **maximum de blocs sensibles deux à deux disjoints** — le plus grand cardinal d'une famille de blocs non vides, chacun sensible ($F(x \oplus \chi_S) \ne F(x)$), et dont deux membres distincts ne se chevauchent jamais. Ce n'est **pas** le compte brut des blocs sensibles : au sommet nul de `or3` (OR sur trois coordonnées), les sept blocs non vides sont tous sensibles et la valeur standard est **3** — les trois singletons (`or3_blockSensitivity`, mesuré en 3.1) ;
- **`orOfAnds`** — l'exemple séparateur en dimension 4 : $(x_0 \wedge x_1) \vee (x_2 \wedge x_3)$.

Un singleton est un bloc : $s \le bs$. La question est de savoir si l'inégalité peut être **stricte** — et `orOfAnds` y répond au sommet nul $x = 0000$ : aucune coordonnée seule n'allume $F$ ($s = 0$), mais le bloc $\{0, 1\}$ l'allume ($bs \ge 1$).


### 3.1 Expérience numérique — `#eval` natif

Le calcul se fait **en Lean**, sur la définition **même** du lac : chaque `#eval blockSensitivity ...` ci-dessous appelle `Sensitivity.blockSensitivity` — le maximum sur les familles disjointes — et non une réimplémentation de comptage. Une subtilité honnête, visible dans le code : `Finset.toList` est non calculable dans Mathlib ; les **profils** de $s$ et $bs$ sur le cube se lisent donc en **histogrammes** — pour chaque valeur $k$, combien de sommets ont $s(x) = k$ (resp. $bs(x) = k$).

Six mesures : la sensibilité locale de `orOfAnds` au sommet nul ; l'histogramme de $s$ sur les 16 sommets de $Q_4$ ; la sensibilité de bloc au sommet nul ; l'histogramme de $bs$ ; puis, sur `or3` au sommet nul, la valeur standard de $bs$ **contre** le compte brut des blocs sensibles — le témoin qui distingue les deux lectures de la définition :


In [6]:
#eval sensitivity orOfAnds (fun _ => false)

#eval ((List.range 3).map fun k =>
  ((Finset.univ : Finset (Q 4)).filter (fun x => sensitivity orOfAnds x = k)).card)

#eval blockSensitivity orOfAnds (fun _ => false)

#eval ((List.range 4).map fun k =>
  ((Finset.univ : Finset (Q 4)).filter (fun x => blockSensitivity orOfAnds x = k)).card)

#eval blockSensitivity or3 (fun _ => false)

#eval ((Finset.univ : Finset (Finset (Fin 3))).filter
    (fun S : Finset (Fin 3) => 0 < S.card ∧
      or3 (flipBlock (fun _ => false) S) ≠ or3 (fun _ => false))).card


#eval sensitivity orOfAnds (fun _ => false)
─────▶  0

#eval ((List.range 3).map fun k =>
─────▶  [2, 4, 10]
  ((Finset.univ : Finset (Q 4)).filter (fun x => sensitivity orOfAnds x = k)).card)

#eval blockSensitivity orOfAnds (fun _ => false)
─────▶  2

#eval ((List.range 4).map fun k =>
─────▶  [0, 0, 16, 0]
  ((Finset.univ : Finset (Q 4)).filter (fun x => blockSensitivity orOfAnds x = k)).card)

#eval blockSensitivity or3 (fun _ => false)
─────▶  3

#eval ((Finset.univ : Finset (Finset (Fin 3))).filter
─────▶  7
    (fun S : Finset (Fin 3) => 0 < S.card ∧
      or3 (flipBlock (fun _ => false) S) ≠ or3 (fun _ => false))).card

--% env 5

Raw input:
{"cmd": "#eval sensitivity orOfAnds (fun _ => false)\n\n#eval ((List.range 3).map fun k =>\n  ((Finset.univ : Finset (Q 4)).filter (fun x => sensitivity orOfAnds x = k)).card)\n\n#eval blockSensitivity orOfAnds (fun _ => false)\n\n#eval ((List.range 4).map fun k =>\n  ((Finset.univ : Finset (Q 4)).filter (fun x => blockSensitivity orOfAnds x = k)).card)\n\n#eval blockSensitivity or3 (fun _ => false)\n\n#eval ((Finset.univ : Finset (Finset (Fin 3))).filter\n    (fun S : Finset (Fin 3) => 0 < S.card \u2227\n      or3 (flipBlock (fun _ => false) S) \u2260 or3 (fun _ => false))).card\n", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 5},
   "data": "0"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "[2, 4, 10]"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "2"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "[0, 0, 16, 0]"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "3"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 5},
   "data": "7"}],
 "env": 5}

### Lecture du résultat

- Au sommet nul $0000$ : $s = 0$ et $bs = 2$. Les blocs qui allument la fonction contiennent la paire $\{0,1\}$ ou la paire $\{2,3\}$ — ils sont nombreux (7 en tout), mais **deux au maximum peuvent être disjoints** : $\{0,1\}$ et $\{2,3\}$. La séparation stricte de `orOfAnds_strict_separation`, calculée sur la définition standard.
- Histogramme de $s$ (pour $k = 0, 1, 2$) : `[2, 4, 10]` — $s = 0$ en exactement deux sommets, le sommet nul $0000$ **et** le sommet tout-un $1111$ (retourner une coordonnée y préserve la valeur : l'autre paire reste saturée) ; $s$ culmine à **2** sur dix sommets. La sensibilité de `orOfAnds` est partout petite.
- Histogramme de $bs$ (pour $k = 0..3$) : `[0, 0, 16, 0]` — $bs = 2$ sur **tous** les sommets. Partout, l'exemple offre exactement deux « marches » disjointes : soit les deux paires à saturer en bloc (où $F = 0$), soit les deux directions de casser les paires saturées (où $F = 1$). L'inégalité $s \le bs$ tient partout, avec **égalité** sur les dix sommets où $s = 2$ : sur cet exemple, elle est serrée.
- Le témoin `or3` au sommet nul : $bs = 3$ quand le compte brut des blocs sensibles vaut **7**. La définition standard lit le maximum de blocs disjoints (les trois singletons), pas l'énumération — c'est exactement la différence entre les deux lectures, exécutée.

C'est **la même expérience** que celle qu'un lecteur ferait en Python par force brute (énumérer $2^4$ sommets et $2^4$ blocs), mais exécutée par le compilateur Lean sur les définitions **mêmes** du lac — une seule source de vérité, pas une réimplémentation.


In [7]:
#check Sensitivity.orOfAnds_sensitivity_zero
#check Sensitivity.orOfAnds_blockSensitivity_pos
#check Sensitivity.orOfAnds_strict_separation
#check Sensitivity.or3_blockSensitivity
#check Sensitivity.or3_sensitive_blocks_card

#print axioms Sensitivity.orOfAnds_strict_separation


#check Sensitivity.orOfAnds_sensitivity_zero
──────▶  Sensitivity.orOfAnds_sensitivity_zero : (sensitivity orOfAnds fun x => false) = 0
#check Sensitivity.orOfAnds_blockSensitivity_pos
──────▶  Sensitivity.orOfAnds_blockSensitivity_pos : 0 < blockSensitivity orOfAnds fun x => false
#check Sensitivity.orOfAnds_strict_separation
──────▶  Sensitivity.orOfAnds_strict_separation :
  (sensitivity orOfAnds fun x => false) < blockSensitivity orOfAnds fun x => false
#check Sensitivity.or3_blockSensitivity
──────▶  Sensitivity.or3_blockSensitivity : (blockSensitivity or3 fun x => false) = 3
#check Sensitivity.or3_sensitive_blocks_card
──────▶  Sensitivity.or3_sensitive_blocks_card :
  {S | S.Nonempty ∧ or3 (flipBlock (fun x => false) S) ≠ or3 fun x => false}.card = 7

#print axioms Sensitivity.orOfAnds_strict_separation
──────▶  'Sensitivity.orOfAnds_strict_separation' depends on axioms: [propext, Classical.choice, Quot.sound]

--% env 6

Raw input:
{"cmd": "#check Sensitivity.orOfAnds_sensitivity_zero\n#check Sensitivity.orOfAnds_blockSensitivity_pos\n#check Sensitivity.orOfAnds_strict_separation\n#check Sensitivity.or3_blockSensitivity\n#check Sensitivity.or3_sensitive_blocks_card\n\n#print axioms Sensitivity.orOfAnds_strict_separation\n", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "Sensitivity.orOfAnds_sensitivity_zero : (sensitivity orOfAnds fun x => false) = 0"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sensitivity.orOfAnds_blockSensitivity_pos : 0 < blockSensitivity orOfAnds fun x => false"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Sensitivity.orOfAnds_strict_separation :\n  (sensitivity orOfAnds fun x => false) < blockSensitivity orOfAnds fun x => false"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "Sensitivity.or3_blockSensitivity : (blockSensitivity or3 fun x => false) = 3"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "Sensitivity.or3_sensitive_blocks_card :\n  {S | S.Nonempty ∧ or3 (flipBlock (fun x => false) S) ≠ or3 fun x => false}.card = 7"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "'Sensitivity.orOfAnds_strict_separation' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 6}

### La séparation stricte, formellement

Les trois théorèmes disent exactement ce que le calcul a montré : $s(\texttt{orOfAnds}, 0000) = 0$, $0 < bs(\texttt{orOfAnds}, 0000)$, donc $s < bs$ sur cet exemple. Les théorèmes du témoin `or3` — `or3_blockSensitivity : bs = 3` et `or3_sensitive_blocks_card : compte brut = 7` — énoncent la différence des deux lectures. L'empreinte d'axiomes de `orOfAnds_strict_separation` est **identique à celle du théorème de Huang** (`[propext, Classical.choice, Quot.sound]`) : aucune dette logique nouvelle, le module `Separations` s'ajoute au lac sans en dégrader la garantie.


## 4. Familles 127, 186, 192 — le vocabulaire Fourier

Les trois autres familles parlent un dialecte voisin : sensibilité **moyenne** (127), influences et seuils (186), coefficients de Fourier **linéaires** (192). Notre lac porte exactement ce vocabulaire dans son module `Fourier` — ce sont les objets que ces familles bornent ou réfutent :

In [8]:
#check Sensitivity.χ
#check Sensitivity.fourierCoeff
#check Sensitivity.influence

#check Sensitivity.χ
──────▶  Sensitivity.χ {n : ℕ} (S : Finset (Fin n)) (x : Q n) : ℤ
#check Sensitivity.fourierCoeff
──────▶  Sensitivity.fourierCoeff {n : ℕ} (f : Q n → ℤ) (S : Finset (Fin n)) : ℤ
#check Sensitivity.influence
──────▶  Sensitivity.influence {n : ℕ} (F : Q n → Bool) (i : Fin n) : ℤ
--% env 7

Raw input:
{"cmd": "#check Sensitivity.\u03c7\n#check Sensitivity.fourierCoeff\n#check Sensitivity.influence", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "Sensitivity.χ {n : ℕ} (S : Finset (Fin n)) (x : Q n) : ℤ"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Sensitivity.fourierCoeff {n : ℕ} (f : Q n → ℤ) (S : Finset (Fin n)) : ℤ"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "Sensitivity.influence {n : ℕ} (F : Q n → Bool) (i : Fin n) : ℤ"}],
 "env": 7}

### Lecture — un vocabulaire, des statuts

- **$\chi$** : les caractères de $Q_n$, la base de Fourier du cube ;
- **`fourierCoeff`** : le coefficient $\widehat F(\xi)$ sur le caractère $\xi$ — c'est la quantité que la famille **192** déclare pouvoir dépasser $C\sqrt{\deg f}$ pour tout $C$ ;
- **`influence`** : l'influence d'une coordonnée, $\Pr_{x}[F(x \oplus e_i) \ne F(x)]$ — la brique dont la famille **127** borne la somme (sensibilité moyenne $\le 8d\sqrt n$ pour les fonctions à seuil polynomiales) et dont la famille **186** tire ses largeurs de seuil.

Statut des trois familles : **formalisée** (par les auteurs). Nous exécutons **leurs objets** dans **notre** lac — les définitions ci-dessus sont les nôtres et tournent ; leurs théorèmes restent des citations renvoyées à leurs squelettes.

## 5. Exercices

### Exercice 1 — Le ET est localement maximalement sensible

Définissez `andOf3 : Q 3 → Bool` (le ET de trois bits), puis montrez **par `#eval`** (mêmes patterns que la section 3.1) que sa sensibilité locale au sommet $111$ vaut $3$ — chaque bit est critique — tandis qu'au sommet nul elle vaut $0$. En quel(s) autre(s) sommet(s) la sensibilité est-elle non nulle ? Expliquez pourquoi le ET « concentre » toute sa sensibilité sur le sommet tout-vrai alors que `orOfAnds` la disperse.

### Exercice 2 — Égalité et séparation sur $Q_2$

Sur $Q_2$, trouvez : (a) une fonction où $s(x) = bs(x)$ en **tout** point ; (b) une fonction où $s(x) < bs(x)$ en **tout** point non constant. Vérifiez chaque cas par `#eval` avec la définition `blockSensitivity` du lac (le maximum de blocs disjoints). Le cas (a) trivial (fonction constante) compte-t-il ? Montrez que pour **toute** fonction et **tout** point, $s(x) \le bs(x)$ — retrouvez l'argument singleton de `sensitivity_le_blockSensitivity`.

### Exercice 3 — Jusqu'où la séparation ? (sur papier)

Généralisez `orOfAnds` : $F_k$ = le OU de $k$ blocs AND de $k$ bits chacun, en dimension $k^2$, sur le sommet nul. Montrez sur papier que $s(F_k, 0) = 0$ et $bs(F_k, 0) \ge k$ : la séparation $bs/s$ (au sens du rapport des maxima globaux) peut devenir arbitrairement grande. Ce calcul esquisse **pourquoi** le renforcement du lien $s \leftrightarrow bs$ est une vraie question — celle que la famille 132 tranche à l'échelle 2026 avec son contre-exemple superquadratique. Sur papier toujours : pourquoi l'argument « singleton » ne donne-t-il que $s \le bs$ et jamais mieux ?

## Conclusion

Ce carnet a tenu les trois registres de la digestion Origami :

1. **Le socle nôtre** — le théorème de Huang, prouvé dans notre lac, `#check` et `#print axioms` à l'appui : statut **nôtre**, empreinte `[propext, Classical.choice, Quot.sound]`.
2. **Les familles 2026 citées** — 127, 132, 186, 192 : énoncés repris fidèlement de leurs résumés, statut **formalisée** (par les auteurs), squelettes `lean/docs/NNN.md` du dépôt openai/math en référence — jamais exécutés ni re-prouvés ici.
3. **Notre propre contribution** — le module `Sensitivity.Separations` : définitions $s$, $bs$, `flipBlock`, l'inégalité plancher $s \le bs$ et la séparation stricte explicite en dimension 4, prouvées **et** calculées (`#eval`) sur les définitions mêmes du lac.

**Verdicts SOTA** (règle du dépôt) : **SOTA-OK** pour ce carnet — vrai Lean 4 + Mathlib, vrai lac compilé, vrai `#eval` natif, aucune substitution. **INTRINSIC** pour la re-preuve des familles 2026 : hors d'échelle de ce dépôt — l'existence des squelettes publiés est précisément ce qui rend la citation le geste honnête.

**Références** : lac [`sensitivity_lean`](../sensitivity_lean/) · issue #19898 (Epic Origami) et #19908 (Pli 2) · Huang, *Induced subgraphs of hypercubes and a proof of the sensitivity conjecture*, Annals of Mathematics 190 (2019) · Gotsman & Linial, *On the influence of variables on Boolean functions* (1991) · Friedgut & Kalai (1996) · manuscrits 127/132/186/192 du dépôt openai/math (2026) et leurs squelettes `lean/docs/NNN.md`.